# Setup inicial

---

## Configurações

### Adicionando o JDK

In [232]:
import os
import subprocess
from pathlib import Path

import jdk

JDK_VERSION = "17"
JDK_DIR = Path.cwd().parent.parent.parent / ".jdk"


def get_java_home(base: Path):
    """Procura a pasta que contém bin/java dentro de base."""
    exe = "java.exe" if os.name == "nt" else "java"
    for caminho in base.rglob(f"bin/{exe}"):
        return caminho.parent.parent
    return None


java_home = get_java_home(JDK_DIR) if JDK_DIR.exists() else None

if java_home is None:
    print("Baixando o JDK...")
    jdk.install(JDK_VERSION, path=str(JDK_DIR))
    java_home = get_java_home(JDK_DIR)

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ["PATH"]

print("JAVA_HOME =", java_home)
subprocess.run(["java", "-version"])

JAVA_HOME = /home/mateus/Code/Eng-Dados/apache-spark-minio/.jdk/jdk-17.0.20.1+1


openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment Temurin-17.0.20.1+1 (build 17.0.20.1+1)
OpenJDK 64-Bit Server VM Temurin-17.0.20.1+1 (build 17.0.20.1+1, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

### Puxando variáveis de ambiente

In [233]:
from dotenv import load_dotenv

load_dotenv(override=True)

DB_HOST = os.getenv("HOST")
DB_PORT = os.getenv("PORT")
DB_NAME = os.getenv("DATABASE")
DB_USER = os.getenv("USER")
DB_PASS = os.getenv("PASSWORD")

print(f"Server: {DB_HOST}:{DB_PORT}")
print(f"Database: {DB_NAME}")

Server: 127.0.0.1:5432
Database: delta_games


### Montando a string de conexão e a URL jdbc

In [234]:
import pyodbc

conn_str = (
    "DRIVER={PostgreSQL Unicode};"
    f"SERVER={DB_HOST};"
    f"PORT={DB_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={DB_USER};"
    f"PWD={DB_PASS};"
)

url = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"

props = {
    "user": DB_USER,
    "password": DB_PASS,
    "driver": "org.postgresql.Driver"
}


### Importando o arquivo de DDL.sql

In [235]:
sql_path = Path("../database/DDL.sql")

sql = sql_path.read_text(encoding="utf-8")

with pyodbc.connect(conn_str) as conn:
    cursor = conn.cursor()
    cursor.execute(sql)

print("Tabelas do branco criadas")

del cursor
del conn

Tabelas do branco criadas


### Configurando pyspark

In [236]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("csv_para_banco")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

---

## Inserindo os Datasets

### Lendo o arquivo CSV do metacritic

In [237]:
csv_path = JDK_DIR.parent / "data" / "metacritic_games_not_null.csv"

df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

### Checando se o CSV do metacritic já foi gravado e gravando caso não

In [238]:
exists = spark.read.jdbc(url, "games", properties=props)

if exists.limit(1).count() == 0:
    df.write.jdbc(url=url, table="games", mode="append", properties=props)
else:
    print("Tabela games já gravada anteriormente.")

Tabela games já gravada anteriormente.


### Conferindo os dados gravados

In [239]:
games_df = spark.read.jdbc(url, "games", properties=props)

print(f"Valores inseridos: {games_df.count()}", end="\n" + ("--" * 20) + "\n")

games_df.show(5)

Valores inseridos: 21910
----------------------------------------
+---+---------------+-----------------+------------+---------+----------+--------------------+--------------------+--------------------+
| id|           name|         platform|release_date|metascore|user_score|           developer|           publisher|               genre|
+---+---------------+-----------------+------------+---------+----------+--------------------+--------------------+--------------------+
|  1|          #BLUD|               PC|  2024-06-18|       73|       5.6|     Exit 73 Studios|    ['Humble Games']|    Action Adventure|
|  2|         #DRIVE|iOS (iPhone/iPad)|  2021-02-16|       69|       6.8|    Dariusz Pietrala|['PM Studios Inc....|         Auto Racing|
|  3|         #IDARB|         Xbox One|  2015-01-30|       77|       6.9|Other Ocean Inter...|['Other Ocean Int...|               Party|
|  4|#KILLALLZOMBIES|               PC|  2016-08-10|       53|       3.5|         Beatshapers|['Digerati Distri.

### Filtrando os jogos de PC para relacionar com jogos da steam

In [240]:
from pyspark.sql.functions import col

game_df = spark.read.jdbc(url, "games", properties=props)
game_df = game_df.where(col("platform") == "PC")
game_df.count()

6282

### Lendo o CSV das métricas da steam e filtrando com os jogos inseridos para PC

In [241]:
steam_df = spark.read.csv(
    str(csv_path.parent / "steam_metrics.csv"),
    header=True,
    inferSchema=True
)

steam_df = steam_df.withColumnsRenamed({
    "AppID": "steam_appid",
    "Name": "name",
    "Price": "price",
    "Achievements": "achievements",
    "Positive": "positive_reviews",
    "Negative": "negative_reviews",
    "Average playtime forever": "avg_playtime"
})

metrics_df = steam_df.join(game_df.select("name", "id"), on="name", how="inner")
metrics_df = metrics_df.withColumnRenamed("id", "game_id")
metrics_df.show(5)

+--------------------+-----------+-----+----------------+----------------+------------+------------+-------+
|                name|steam_appid|price|positive_reviews|negative_reviews|achievements|avg_playtime|game_id|
+--------------------+-----------+-----+----------------+----------------+------------+------------+-------+
|               #BLUD|    1921480|14.99|             251|              65|          43|         343|      1|
|     #KILLALLZOMBIES|     303720| 4.99|             242|              67|          10|         329|      4|
|.hack//G.U. Last ...|     525480| 9.99|            3358|             514|          52|        1040|      9|
|           0RBITALIS|     278440| 9.99|             219|              88|          38|         179|     34|
|     10 Second Ninja|     271670| 1.99|             659|             115|          30|         104|     36|
+--------------------+-----------+-----+----------------+----------------+------------+------------+-------+
only showing top 5 

### Checando se o CSV das métricas da steam já foi gravado e gravando caso não

In [242]:
metrics_df = metrics_df.drop("name")

exists = spark.read.jdbc(url, "steam_metrics", properties=props)

if exists.limit(1).count() == 0:
    metrics_df.write.jdbc(url=url, table="steam_metrics", mode="append", properties=props)
else:
    print("Tabela steam_metrics já gravada anteriormente")

Tabela steam_metrics já gravada anteriormente


### Conferindo os dados gravados

In [243]:
spark.read.jdbc(url, "steam_metrics", properties=props).count()

3178

### Gerando um DataFrame padrão com o ID e o NOME

In [244]:
std_df = spark.read.jdbc(url, "games", properties=props).select(col("id").alias("game_id"), col("name"))
std_df.show()

+-------+--------------------+
|game_id|                name|
+-------+--------------------+
|      1|               #BLUD|
|      2|              #DRIVE|
|      3|              #IDARB|
|      4|     #KILLALLZOMBIES|
|      5|     #KILLALLZOMBIES|
|      6|       'Splosion Man|
|      7|            .detuned|
|      8|.hack//G.U. Last ...|
|      9|.hack//G.U. Last ...|
|     10|.hack//G.U. Last ...|
|     11|.hack//G.U. vol. ...|
|     12|.hack//G.U. vol. ...|
|     13|.hack//G.U. vol. ...|
|     14|.hack//Infection ...|
|     15|.hack//Mutation P...|
|     16|.hack//Outbreak P...|
|     17|0 Day Attack on E...|
|     18|         007 Legends|
|     19|         007 Legends|
|     20|         007 Legends|
+-------+--------------------+
only showing top 20 rows


### Lendo o CSV das vendas regionais e unindo com o DataFrame padrão

In [245]:
sales_df = spark.read.csv(
    str(csv_path.parent / "sales.csv"),
    header=True,
    inferSchema=True
)

sales_df = sales_df.withColumnsRenamed({
    "Name": "name",
    "Platform": "platform",
    "NA_Sales": "na_sales",
    "EU_Sales": "eu_sales",
    "JP_Sales": "jp_sales",
    "Other_Sales": "other_sales",
    "Global_Sales": "global_sales"
})

sales_df = sales_df.join(std_df.select("name", "game_id"), on="name", how="inner")

sales_df.show(5)

+--------------------+--------+--------+--------+--------+-----------+------------+-------+
|                name|platform|na_sales|eu_sales|jp_sales|other_sales|global_sales|game_id|
+--------------------+--------+--------+--------+--------+-----------+------------+-------+
|.hack//Infection ...|     PS2|    0.49|    0.38|    0.26|       0.13|        1.27|     14|
|.hack//Mutation P...|     PS2|    0.23|    0.18|     0.2|       0.06|        0.68|     15|
|.hack//Outbreak P...|     PS2|    0.14|    0.11|    0.17|       0.04|        0.46|     16|
|          007 Racing|      PS|     0.3|     0.2|     0.0|       0.03|        0.53|     21|
|007: Quantum of S...|      PC|    0.01|    0.01|     0.0|        0.0|        0.03|     27|
+--------------------+--------+--------+--------+--------+-----------+------------+-------+
only showing top 5 rows


### Checando se o CSV das vendas regionais já foi gravado e gravando caso não

In [246]:
sales_df = sales_df.drop("name", "platform")

exists = spark.read.jdbc(url, "regional_sales", properties=props)

if exists.limit(1).count() == 0:
    sales_df.write.jdbc(url, "regional_sales", mode="append", properties=props)
else:
    print("Tabela regional_sales já gravada anteriormente.")

Tabela regional_sales já gravada anteriormente.


### Conferindo os dados gravados

In [247]:
spark.read.jdbc(url, "regional_sales", properties=props).count()

16608

### Lendo o CSV de HowLongToBeat e unindo com o DataFrame padrão

In [248]:
hltb_df = spark.read.csv(
    str(csv_path.parent / "hltb_corr_values.csv"),
    header=True,
    inferSchema=True
)

hltb_df = hltb_df.withColumnsRenamed({
    "main_story": "main_story_hours",
    "main_plus_sides": "extras_hours",
    "completionist": "completionist_hours"
})

hltb_df = hltb_df.join(std_df.select("name", "game_id"), on="name", how="inner")

hltb_df.show(5)

+-------------+-----+----+--------------------+----------------+------------+-------------------+-------+
|         name|   id|type|            platform|main_story_hours|extras_hours|completionist_hours|game_id|
+-------------+-----+----+--------------------+----------------+------------+-------------------+-------+
|        #BLUD|45648|game|Nintendo Switch, ...|            9.23|       11.09|              17.91|      1|
|'Splosion Man|    1|game|            Xbox 360|             7.8|        9.47|              17.62|      6|
|     .detuned| 9627|game|       PlayStation 3|            0.17|        0.33|               0.41|      7|
|  007 Legends|   10|game|PC, PlayStation 3...|            6.46|        8.21|               9.69|     18|
|  007 Legends|   10|game|PC, PlayStation 3...|            6.46|        8.21|               9.69|     19|
+-------------+-----+----+--------------------+----------------+------------+-------------------+-------+
only showing top 5 rows


### Checando se o CSV de HowLongToBeat já foi gravado e gravando caso não

In [249]:
hltb_df = hltb_df.drop("name", "id", "type", "platform")

exists = spark.read.jdbc(url, "how_long_to_beat", properties=props)

if exists.limit(1).count() == 0:
    hltb_df.write.jdbc(url, "how_long_to_beat", mode="append", properties=props)
else:
    print("Tabela how_long_to_beat já gravada anteriormente.")

Tabela how_long_to_beat já gravada anteriormente.


### Conferindo os dados gravados

In [250]:
spark.read.jdbc(url, "how_long_to_beat", properties=props).count()

13196

### Lendo o CSV dos links de walkthrough

In [251]:
from pyspark.sql import functions as F

links_df = spark.read.csv(
    str(csv_path.parent / "strategywiki_links_walkthrough.csv"),
    header=True,
    inferSchema=True
)

links_df = links_df.dropDuplicates(["jogo"])

links_df.count()

2137

### Unindo o DataFrame links com o DataFrame padrão

In [252]:
std_df = std_df.withColumn("bak", F.lower(F.regexp_replace(F.col("name"), r"\s+", "")))
links_df = links_df.withColumn("bak", F.lower(F.regexp_replace(F.col("jogo"), r"\s+", "")))

links_df = links_df.withColumnRenamed("jogo", "name")

links_df = std_df.join(links_df.select("link", "bak"), on="bak", how="inner")

links_df = links_df.dropDuplicates(["bak"])

links_df.show(5)

+--------------------+-------+--------------------+--------------------+
|                 bak|game_id|                name|                link|
+--------------------+-------+--------------------+--------------------+
|      aboyandhisblob|    144|  A Boy and His Blob|https://strategyw...|
|acecombat04:shatt...|    311|Ace Combat 04: Sh...|https://strategyw...|
|acecombat6:fireso...|    314|Ace Combat 6: Fir...|https://strategyw...|
|advancewars2:blac...|    366|Advance Wars 2: B...|https://strategyw...|
|advancewars:dayso...|    367|Advance Wars: Day...|https://strategyw...|
+--------------------+-------+--------------------+--------------------+
only showing top 5 rows


### Checando se o CSV de links de Walkthroughs já foi gravado e gravando caso não

In [253]:
links_df = links_df.select("game_id", "link")
links_df = links_df.withColumnRenamed("link", "url")

exists = spark.read.jdbc(url, "walkthrough_links", properties=props)

if exists.limit(1).count() == 0:
    links_df.write.jdbc(url, "walkthrough_links", mode="append", properties=props)
else:
    print("Tabela walkthrough_links já gravada anteriormente.")

links_df.show(5)

Tabela walkthrough_links já gravada anteriormente.
+-------+--------------------+
|game_id|                 url|
+-------+--------------------+
|    144|https://strategyw...|
|    311|https://strategyw...|
|    314|https://strategyw...|
|    366|https://strategyw...|
|    367|https://strategyw...|
+-------+--------------------+
only showing top 5 rows


### Listando o número de registros das tabelas

In [254]:
games_df = spark.read.jdbc(url, "games", properties=props).count()
steam_df = spark.read.jdbc(url, "steam_metrics", properties=props).count()
hltb_df = spark.read.jdbc(url, "how_long_to_beat", properties=props).count()
sales_df = spark.read.jdbc(url, "regional_sales", properties=props).count()
links_df = spark.read.jdbc(url, "walkthrough_links", properties=props).count()

print(f"{'Games:':<20} {games_df:>8} registros")
print(f"{'Métricas Steam:':<20} {steam_df:>8} registros")
print(f"{'HowLongToBeat:':<20} {hltb_df:>8} registros")
print(f"{'Vendas Regionais:':<20} {sales_df:>8} registros")
print(f"{'Links de Detonados:':<20} {links_df:>8} registros")

Games:                  21910 registros
Métricas Steam:          3178 registros
HowLongToBeat:          13196 registros
Vendas Regionais:       16608 registros
Links de Detonados:       688 registros


In [255]:
if spark: del spark
if df: del df
if metrics_df: del metrics_df
if std_df: del std_df
if game_df: del game_df
if games_df: del games_df
if steam_df: del steam_df
if hltb_df: del hltb_df
if sales_df: del sales_df
if links_df: del links_df
if exists: del exists
if csv_path: del csv_path
if sql: del sql
if sql_path: del sql_path